In [ ]:
import os
import shutil
import yaml
import numpy as np
import collections
import pandas as pd
from pathlib import Path
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from ultralytics import YOLO

BASE_PATH = "./all"
IMAGES_DIR = Path(BASE_PATH) / "images"
LABELS_DIR = Path(BASE_PATH) / "labels"

WORKING_DIR = Path("split_original")

In [ ]:
# find all classes present in a label file
def get_present_classes(label_path):
    if not os.path.exists(label_path):
        return []
    with open(label_path, 'r') as f:
        # format: <class_id> ...
        return list(set([int(line.split()[0]) for line in f.readlines()]))

images = sorted([f for f in os.listdir(IMAGES_DIR) if f.endswith(('.jpg', '.png'))])
num_classes = 3  # 0: Fullripe, 1: Semiripe, 2: Unripe

# multi-label matrix to check presence of each label in an image
y_matrix = np.zeros((len(images), num_classes))
for i, img_name in enumerate(images):
    label_path = LABELS_DIR / (img_name.rsplit('.', 1)[0] + '.txt')
    classes = get_present_classes(label_path)
    for c in classes:
        y_matrix[i, c] = 1

X = np.array(images)

# split 70% for train, 30% for val-test
msss_train = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(msss_train.split(X, y_matrix))

X_train, y_temp, X_temp = X[train_idx], y_matrix[temp_idx], X[temp_idx]

# split the remaining 30% to val and test
msss_test = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(msss_test.split(X_temp, y_temp))

X_val, X_test = X_temp[val_idx], X_temp[test_idx]

print(f"Dataset split -> Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
# create directories
for split in ['train', 'val', 'test']:
    os.makedirs(WORKING_DIR / 'images' / split, exist_ok=True)
    os.makedirs(WORKING_DIR / 'labels' / split, exist_ok=True)

def populate_split(file_list, split_name):
    for img_name in file_list:
        label_name = img_name.rsplit('.', 1)[0] + '.txt'
        
        # copy image
        src_img = IMAGES_DIR / img_name
        if src_img.exists():
            shutil.copy(src_img, WORKING_DIR / 'images' / split_name / img_name)
            
        # copy label
        src_label = LABELS_DIR / label_name
        if src_label.exists():
            shutil.copy(src_label, WORKING_DIR / 'labels' / split_name / label_name)

populate_split(X_train, 'train')
populate_split(X_val, 'val')
populate_split(X_test, 'test')

# generate dataset.yaml
yaml_content = {
    'train': str(WORKING_DIR / 'images' / 'train'),
    'val': str(WORKING_DIR / 'images' / 'val'),
    'test': str(WORKING_DIR / 'images' / 'test'),
    'nc': 3,
    'names': ['Fullripe', 'Semiripe', 'Unripe']
}

yaml_path = WORKING_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)
    
print("YOLO dataset structure and data.yaml created successfully.")

In [ ]:
splits = ['train', 'val', 'test']
class_names = {0: 'Fullripe', 1: 'Semiripe', 2: 'Unripe'}

distribution_data = []
# count class distribution
for split in splits:
    split_dir = WORKING_DIR / 'labels' / split
    counter = collections.Counter()
    
    if split_dir.exists():
        for label_file in split_dir.glob('*.txt'):
            with open(label_file, 'r') as f:
                for line in f:
                    cls_id = int(line.split()[0])
                    counter[cls_id] += 1
                    
    distribution_data.append({
        'Split': split.capitalize(),
        'Fullripe': counter[0],
        'Semiripe': counter[1],
        'Unripe': counter[2],
        'Total BBoxes': sum(counter.values())
    })

df_dist = pd.DataFrame(distribution_data)
print("Bounding Box Class Distribution per Split:")
print("-" * 50)
print(df_dist.to_string(index=False))

In [ ]:
import cv2
import random
import matplotlib.pyplot as plt

train_img_dir = WORKING_DIR / 'images' / 'train'
train_lbl_dir = WORKING_DIR / 'labels' / 'train'

classes = {0: 'Fullripe', 1: 'Semiripe', 2: 'Unripe'}
colors = {0: (255, 0, 0), 1: (255, 165, 0), 2: (0, 255, 0)}
# function to visualize some samples
def plot_ground_truth_polygons(num_samples=3):
    all_images = list(train_img_dir.glob('*.jpg')) + list(train_img_dir.glob('*.png'))
    samples = random.sample(all_images, min(num_samples, len(all_images)))
    
    fig, axes = plt.subplots(1, num_samples, figsize=(18, 6))
    if num_samples == 1: axes = [axes]
    
    for ax, img_path in zip(axes, samples):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        
        lbl_path = train_lbl_dir / (img_path.stem + '.txt')
        
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    cls_id = int(parts[0])
                    color = colors.get(cls_id, (255, 255, 255))
                    
                    if len(parts) == 5:
                        x_c, y_c, bw, bh = parts[1:]
                        x1, y1 = int((x_c - bw/2) * w), int((y_c - bh/2) * h)
                        x2, y2 = int((x_c + bw/2) * w), int((y_c + bh/2) * h)
                        cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
                        cv2.putText(img, classes[cls_id], (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
                        
                    elif len(parts) > 5:
                        coords = np.array(parts[1:]).reshape(-1, 2)
                        
                        coords[:, 0] *= w
                        coords[:, 1] *= h
                        coords = np.int32(coords)
                        
                        cv2.polylines(img, [coords], isClosed=True, color=color, thickness=3)
                        
                        x_min, y_min = coords[:, 0].min(), coords[:, 1].min()
                        x_max, y_max = coords[:, 0].max(), coords[:, 1].max()
                        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), color, 1)
                        cv2.putText(img, classes[cls_id], (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
                        
        ax.imshow(img)
        ax.set_title(img_path.name)
        ax.axis('off')
        
    plt.tight_layout()
    plt.show()

plot_ground_truth_polygons(num_samples=3)

In [ ]:
# The hue values to test
hue_values = [0.0, 0.03, 0.05, 0.1, 0.15, 0.3]
results_list = []

print("Starting full Grid Search with Early Stopping...")

for h_val in hue_values:
    print(f"\n--- Training with hsv_h = {h_val} ---")
    
    model = YOLO('yolov8s.pt')
    
    # high epochs and patience to find let it converge
    results = model.train(
        data=str(yaml_path),     
        epochs=150,            
        patience=25,           
        imgsz=640,
        batch=16,
        device=0,
        hsv_h=h_val,             
        hsv_s=0.7,               
        hsv_v=0.4,               
        project='runs/grid_search',
        name=f'hsv_{h_val}'      
    )
    
    run_dir = f'runs/grid_search/hsv_{h_val}'
    csv_path = os.path.join(run_dir, 'results.csv')
    
    # find the convergence epoch
    best_epoch = "Unknown"
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        
        best_epoch_idx = df['metrics/mAP50(B)'].idxmax() 
        best_epoch = df.loc[best_epoch_idx, 'epoch']
    
    results_list.append({
        'Hue Jitter (hsv_h)': h_val,
        'Best Val mAP@50': results.box.map50,
        'Optimal Epoch': best_epoch
    })

print("\nGrid Search Complete!")
df_results = pd.DataFrame(results_list)
print("-" * 60)
print(df_results.to_string(index=False))
print(results_list)

In [ ]:
# new yaml just for testing
yaml_final_content = {
    'train': [
        str(WORKING_DIR / 'images' / 'train'),
        str(WORKING_DIR / 'images' / 'val')
    ],
    'val': str(WORKING_DIR / 'images' / 'test'),
    'nc': 3,
    'names': ['Fullripe', 'Semiripe', 'Unripe']
}

yaml_final_path = WORKING_DIR / 'data_final.yaml'
with open(yaml_final_path, 'w') as f:
    yaml.dump(yaml_final_content, f, default_flow_style=False)
    
print("data_final.yaml created for Train+Val combined training.")

In [ ]:
# final baseline model
final_model = YOLO('yolov8s.pt')

final_results = final_model.train(
    data=str(yaml_final_path),     
    epochs=65,               
    imgsz=640,
    batch=16,
    device=0,
    hsv_h=0.03,             
    hsv_s=0.7,              
    hsv_v=0.4,  
    patience=0,
    project='runs',
    name='yolov8s_final'
)

# baseline metrics
print(f"Final Test Mean Precision (mP): {final_results.box.mp:.4f}")
print(f"Final Test Mean Recall (mR):    {final_results.box.mr:.4f}")
print(f"Final Test mAP@50:              {final_results.box.map50:.4f}")
print(f"Final Test mAP@50-95:           {final_results.box.map:.4f}")

In [ ]:
val_img_dir = WORKING_DIR / 'images' / 'val'
val_lbl_dir = WORKING_DIR / 'labels' / 'val'

classes = {0: 'Fullripe', 1: 'Semiripe', 2: 'Unripe'}
colors = {0: (255, 0, 0), 1: (255, 165, 0), 2: (0, 255, 0)}

# load the trained model
model = YOLO('runs/yolov8s_final/weights/best.pt')

def compare_val_predictions(
    model,
    val_img_dir,
    val_lbl_dir,
    num_samples=3,
    conf=0.25,
    iou=0.60,
    classes=None,
    colors=None,
    seed=None,
    image_extensions=("*.jpg", "*.jpeg", "*.png"),
    figsize=(16, 8),
):
    val_img_dir = Path(val_img_dir)
    val_lbl_dir = Path(val_lbl_dir)

    classes = classes or {0: "Fullripe", 1: "Semiripe", 2: "Unripe"}
    colors = colors or {0: (255, 0, 0), 1: (255, 165, 0), 2: (0, 255, 0)}

    if not val_img_dir.exists():
        raise FileNotFoundError(f"Validation image directory not found: {val_img_dir}")
    if not val_lbl_dir.exists():
        raise FileNotFoundError(f"Validation label directory not found: {val_lbl_dir}")
    if num_samples <= 0:
        raise ValueError("num_samples must be greater than 0.")

    all_images = []
    for pattern in image_extensions:
        all_images.extend(val_img_dir.glob(pattern))
    all_images = sorted(set(all_images))

    if not all_images:
        raise ValueError(f"No validation images found in: {val_img_dir}")

    rng = random.Random(seed)
    samples = rng.sample(all_images, min(num_samples, len(all_images)))

    for img_path in samples:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"Warning: could not read image '{img_path.name}'. Skipping.")
            plt.close(fig)
            continue

        img_gt = cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB)
        img_pred = cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB)
        h, w = img_gt.shape[:2]

        lbl_path = val_lbl_dir / f"{img_path.stem}.txt"

        if lbl_path.exists():
            with open(lbl_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if len(parts) < 5:
                        continue

                    cls_id = int(parts[0])
                    color = colors.get(cls_id, (255, 255, 255))
                    class_name = classes.get(cls_id, f"class_{cls_id}")

                    if len(parts) == 5:
                        x_c, y_c, bw, bh = parts[1:]
                        x1 = int((x_c - bw / 2) * w)
                        y1 = int((y_c - bh / 2) * h)
                        x2 = int((x_c + bw / 2) * w)
                        y2 = int((y_c + bh / 2) * h)
                    else:
                        coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
                        coords[:, 0] *= w
                        coords[:, 1] *= h
                        coords = coords.astype(np.int32)

                        cv2.polylines(img_gt, [coords], isClosed=True, color=color, thickness=2)
                        x1, y1 = coords[:, 0].min(), coords[:, 1].min()
                        x2, y2 = coords[:, 0].max(), coords[:, 1].max()

                    cv2.rectangle(img_gt, (x1, y1), (x2, y2), color, 3)
                    cv2.putText(
                        img_gt,
                        class_name,
                        (x1, max(20, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        color,
                        2,
                    )

        results = model.predict(source=str(img_path), conf=conf, iou=iou, verbose=False)

        if results and len(results[0].boxes) > 0:
            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                score = float(box.conf[0])
                cls_id = int(box.cls[0])

                color = colors.get(cls_id, (255, 255, 255))
                class_name = classes.get(cls_id, f"class_{cls_id}")
                label = f"{class_name} {score:.2f}"

                cv2.rectangle(img_pred, (x1, y1), (x2, y2), color, 3)
                cv2.putText(
                    img_pred,
                    label,
                    (x1, max(20, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2,
                )

        ax1.imshow(img_gt)
        ax1.set_title(f"Ground Truth: {img_path.name}")
        ax1.axis("off")

        ax2.imshow(img_pred)
        ax2.set_title(f"Prediction: {img_path.name} (conf={conf}, iou={iou})")
        ax2.axis("off")

        plt.tight_layout()
        plt.show()

compare_val_predictions(model=model, num_samples=5)

In [ ]:
# trying yolo26
model_26 = YOLO('yolo26s.pt')

# finding an optimal epoch count
results_26 = model_26.train(
    data=str(yaml_path),     
    epochs=150,
    patience=25,
    imgsz=640,
    batch=16,
    device=0,
    hsv_h=0.03,             
    hsv_s=0.7,               
    hsv_v=0.4,               
    project='runs',
    name=f'yolo26_hpo' 
)

In [ ]:
# final run for yolo26
model_26_final = YOLO('yolo26s.pt')

# using optimized hyperparameters
results_26 = model_26_final.train(
    data=yaml_final_path,
    epochs=118,              
    imgsz=640,              
    batch=16,                 
    hsv_h=0.03,
    optimizer='auto',
    patience=0,
    device=0,
    project='runs',
    name='yolo26s_final'
)

# final metrics on the test set
print(f"YOLO26s Test Mean Precision (mP): {results_26.box.mp:.4f}")
print(f"YOLO26s Test Mean Recall (mR):    {results_26.box.mr:.4f}")
print(f"YOLO26s Test mAP@50:              {results_26.box.map50:.4f}")
print(f"YOLO26s Test mAP@50-95:           {results_26.box.map:.4f}")

In [ ]:
val_img_dir = WORKING_DIR / 'images' / 'val'
val_lbl_dir = WORKING_DIR / 'labels' / 'val'

# Class mapping and colors
classes = {0: 'Fullripe', 1: 'Semiripe', 2: 'Unripe'}
colors = {0: (255, 0, 0), 1: (255, 165, 0), 2: (0, 255, 0)}

# Load your trained baseline model
model = YOLO('runs/yolo26s_final/weights/best.pt')

# Run the comparison
compare_val_predictions(model=model, num_samples=5)